# Combined Strategies Experiment — SA1 Architecture

Runs 4 optimization combinations on **SA1** (Router → Analyze → Extract → Justify, all `o4-mini`, Router group):

| Step | Label | What it does |
|---|---|---|
| 1 | **Baseline (cold)** | First HTTP call — genuine cold start, full 4-stage pipeline |
| 2 | **+Parallel** | Extract and Justify run concurrently (ThreadPoolExecutor) |
| 3 | **+Cache (miss)** | Cache lookup first; miss → run pipeline, save result |
| 4 | **+Cache (hit)** | Same query → instant hit, zero LLM calls |

**No vLLM. No Qwen. Results saved to a new timestamped folder — previous runs untouched.**

In [ ]:
# ── Cell 1: Setup ─────────────────────────────────────────────────────────────
import sys, os, json, time, hashlib, concurrent.futures
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.metrics.pairwise import cosine_similarity

from dotenv import load_dotenv
load_dotenv(Path('/Users/srujanakadambari/Desktop/ffresh thesis/data-to-visual/.env'), override=True)

from openai import OpenAI
import instructor

PROJECT_ROOT = Path('/Users/srujanakadambari/Desktop/ffresh thesis/data-to-visual/data-to-visual-nicos-branch')
FINAL_FOLDER = PROJECT_ROOT / 'FINAL FOLDER'
os.chdir(PROJECT_ROOT)
for p in [str(PROJECT_ROOT), str(FINAL_FOLDER)]:
    if p not in sys.path:
        sys.path.insert(0, p)

from retrieve_data import retrieve_data
from prompts.default import (
    SYSTEM_PROMPT, DATA_ANALYSIS_PROMPT,
    CHART_CONFIGURATION_PROMPT, CREATE_CHART_TYPE_JUSTIFICATION_PROMPT,
)
from response_models.default import VisualizationConfig, DataAnalysis, ChartTypeJustification, ChartType
from visualization_from_template import generate_from_template

API_KEY = os.getenv('OPENAI_API_KEY', '')
assert API_KEY, 'OPENAI_API_KEY not found in .env'

# LLM client — uses whatever endpoint .env sets (custom proxy is fine for chat)
# OPENAI_BASE_URL in .env is broken — force correct endpoint for all clients
OPENAI_API_BASE = 'https://api.openai.com/v1'
llm_client  = OpenAI(api_key=API_KEY, base_url=OPENAI_API_BASE)
inst_client = instructor.from_openai(OpenAI(api_key=API_KEY, base_url=OPENAI_API_BASE))

# Embedding client — ALWAYS standard OpenAI (custom proxy doesn't support /embeddings)
emb_client  = OpenAI(api_key=API_KEY, base_url=OPENAI_API_BASE)

MD_TABLE = retrieve_data(None, type='test')
QUERY    = 'Wieviel Umsatz hatte Teckentrup im Bereich JVA/CO in den Jahren 2021 bis 2024?Provide a detailed analysis and a comprehensive visualisation.'

RUN_TS   = datetime.now().strftime('%Y%m%d_%H%M%S')
OUT_DIR  = FINAL_FOLDER / f'combined_SA1_{RUN_TS}'
OUT_DIR.mkdir(parents=True, exist_ok=True)
(OUT_DIR / 'renders').mkdir(exist_ok=True)

CACHE_DIR       = FINAL_FOLDER / 'bench_cache'
CACHE_DIR.mkdir(exist_ok=True)

SCENARIO_ID     = 'SA1'
MODEL           = 'o4-mini'
EMBEDDING_MODEL = 'text-embedding-3-small'
SEM_THRESHOLD   = 0.88
ROUTER_PREVIEW  = 500

ROUTER_TOOLS = [{
    'type': 'function',
    'function': {
        'name': 'generate_visualization',
        'description': 'Analyze the data and generate a visualization',
        'parameters': {
            'type': 'object',
            'properties': {
                'data':       {'type': 'string', 'description': 'The data as a markdown table'},
                'user_query': {'type': 'string', 'description': "The user's query"},
            },
            'required': ['data', 'user_query'],
        },
    },
}]

print('Setup OK')
print(f'  Scenario : SA1 — {MODEL} x4 stages')
print(f'  LLM base : {llm_client.base_url}')
print(f'  Emb base : {emb_client.base_url}')
print(f'  Query    : {QUERY}')
print(f'  Output   : {OUT_DIR.name}')


In [ ]:
# ── Cell 2: Pipeline stage functions ──────────────────────────────────────────

def _usage(resp):
    u = getattr(resp, 'usage', None)
    if not u: return {}
    return {
        'prompt_tokens':     getattr(u, 'prompt_tokens', 0) or 0,
        'completion_tokens': getattr(u, 'completion_tokens', 0) or 0,
        'total_tokens':      getattr(u, 'total_tokens', 0) or 0,
    }

def step_router(query, md_table):
    messages = [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user',   'content': f"{query}\n\nData (preview):\n{md_table[:ROUTER_PREVIEW]}"},
    ]
    t0   = time.perf_counter()
    resp = llm_client.chat.completions.create(
        model=MODEL, messages=messages,
        tools=ROUTER_TOOLS, tool_choice='required',
    )
    return time.perf_counter() - t0, _usage(resp)

def step_analyze(query, md_table):
    prompt = DATA_ANALYSIS_PROMPT.format(data=md_table, query=query)
    t0     = time.perf_counter()
    resp, completion = inst_client.chat.completions.create_with_completion(
        model=MODEL,
        messages=[{'role': 'user', 'content': prompt}],
        response_model=DataAnalysis,
    )
    return resp.analysis, time.perf_counter() - t0, _usage(completion)

def step_extract(analysis, md_table):
    prompt = CHART_CONFIGURATION_PROMPT.format(data=md_table, analysis=analysis)
    t0     = time.perf_counter()
    resp, completion = inst_client.chat.completions.create_with_completion(
        model=MODEL,
        messages=[{'role': 'user', 'content': prompt}],
        response_model=VisualizationConfig,
    )
    cfg = resp.model_dump()
    cfg['charttype'] = cfg['charttype'].value if hasattr(cfg['charttype'], 'value') else str(cfg['charttype'])
    return cfg, time.perf_counter() - t0, _usage(completion)

def step_justify(analysis, md_table):
    charttypes = {ct.name for ct in ChartType}
    prompt = CREATE_CHART_TYPE_JUSTIFICATION_PROMPT.format(
        charttypes=charttypes, analysis=analysis, data=md_table)
    t0   = time.perf_counter()
    resp, completion = inst_client.chat.completions.create_with_completion(
        model=MODEL,
        messages=[{'role': 'user', 'content': prompt}],
        response_model=ChartTypeJustification,
    )
    ct = resp.chart_type.value if hasattr(resp.chart_type, 'value') else str(resp.chart_type)
    return ct, time.perf_counter() - t0, _usage(completion)

def run_s0_pipeline(query, md_table, parallel=False):
    timings, tokens = {}, {}

    t, tok = step_router(query, md_table)
    timings['router'] = t
    for k, v in tok.items(): tokens[k] = tokens.get(k, 0) + v

    analysis, t, tok = step_analyze(query, md_table)
    timings['analyze'] = t
    for k, v in tok.items(): tokens[k] = tokens.get(k, 0) + v

    if parallel:
        t0 = time.perf_counter()
        with concurrent.futures.ThreadPoolExecutor(max_workers=2) as ex:
            f_ext = ex.submit(step_extract, analysis, md_table)
            f_jus = ex.submit(step_justify, analysis, md_table)
            cfg, _, tok_e       = f_ext.result()
            charttype, _, tok_j = f_jus.result()
        cfg['charttype'] = charttype
        timings['extract||justify'] = time.perf_counter() - t0
        for tok in [tok_e, tok_j]:
            for k, v in tok.items(): tokens[k] = tokens.get(k, 0) + v
    else:
        cfg, t, tok = step_extract(analysis, md_table)
        timings['extract'] = t
        for k, v in tok.items(): tokens[k] = tokens.get(k, 0) + v
        charttype, t, tok = step_justify(analysis, md_table)
        cfg['charttype'] = charttype
        timings['justify'] = t
        for k, v in tok.items(): tokens[k] = tokens.get(k, 0) + v

    return cfg, timings, tokens

print('Pipeline ready — S0: Router + Analyze + Extract + Justify (o4-mini)')


In [ ]:
# ── Cell 3: Quality validator ──────────────────────────────────────────────────
VALID_CT = {'line', 'bar', 'pie', 'scatter', 'column', 'stackedbar'}

def validate(cfg, render=True, tag='run', save_dir=None):
    r = {
        'schema_complete': False, 'valid_charttype': False, 'has_data': False,
        'has_title': False, 'has_axes': False, 'renderable': False,
        'has_annotations': False, 'data_consistency': False,
        'quality_score': 0, 'passed': False, 'error': None,
        'n_data_groups': 0, 'n_annotations': 0, 'title_text': '', 'charttype': ''
    }
    if cfg is None: r['error'] = 'null config'; return r
    if hasattr(cfg, 'model_dump'): cfg = cfg.model_dump()

    r['schema_complete'] = all(cfg.get(f) for f in ['titlename','charttype','xlabel','ylabel','data'])
    ct = cfg.get('charttype', '')
    if hasattr(ct, 'value'): ct = ct.value
    r['valid_charttype'] = str(ct).lower().strip() in VALID_CT
    r['charttype'] = str(ct)
    data = cfg.get('data') or []
    r['has_data'] = len(data) > 0
    r['n_data_groups'] = len(data)
    r['has_title'] = bool(str(cfg.get('titlename', '')).strip())
    r['title_text'] = str(cfg.get('titlename', ''))[:60]
    r['has_axes'] = bool(cfg.get('xlabel')) and bool(cfg.get('ylabel'))
    consistent = True
    for g in data:
        d = g if isinstance(g, dict) else (g.model_dump() if hasattr(g, 'model_dump') else {})
        xd, yd = d.get('x_data', []) or [], d.get('y_data', []) or []
        if not xd or not yd or len(xd) != len(yd): consistent = False; break
    r['data_consistency'] = consistent
    annos = cfg.get('annotations') or []
    r['has_annotations'] = len(annos) > 0
    r['n_annotations'] = len(annos)

    if render and r['has_data'] and r['valid_charttype']:
        try:
            import copy
            rc = copy.deepcopy(cfg)
            xt = rc.get('x_ticks', []) or []; xl = rc.get('x_tick_label', []) or []
            if len(xl) != len(xt): n = min(len(xt),len(xl)); rc['x_ticks']=xt[:n]; rc['x_tick_label']=xl[:n]
            yt = rc.get('y_ticks', []) or []; yl = rc.get('y_tick_label', []) or []
            if len(yl) != len(yt): n = min(len(yt),len(yl)); rc['y_ticks']=yt[:n]; rc['y_tick_label']=yl[:n]
            _show = plt.show; plt.show = lambda *a, **k: None
            generate_from_template(rc)
            if save_dir:
                plt.savefig(Path(save_dir) / f'{tag}.png', dpi=120, bbox_inches='tight')
            plt.show = _show; plt.close('all')
            r['renderable'] = True
        except Exception as e:
            r['error'] = f'render: {str(e)[:80]}'
            plt.close('all')
            try: plt.show = _show
            except: pass

    checks = ['schema_complete','valid_charttype','has_data','has_title',
               'has_axes','renderable','has_annotations','data_consistency']
    r['quality_score'] = sum(r[c] for c in checks)
    r['passed'] = r['quality_score'] >= 5
    return r

print('Quality validator ready (8 checks)')


In [ ]:
# ── Cell 4: Semantic cache helpers ─────────────────────────────────────────────
CACHE_INDEX = CACHE_DIR / 'cache_index.json'

def load_index():
    return json.loads(CACHE_INDEX.read_text('utf-8')) if CACHE_INDEX.exists() else []

def save_index(idx):
    CACHE_INDEX.write_text(json.dumps(idx, indent=2, ensure_ascii=False), 'utf-8')

def embed(text):
    # Uses emb_client (standard OpenAI) — NOT llm_client which may point to custom proxy
    resp = emb_client.embeddings.create(input=[text], model=EMBEDDING_MODEL)
    return np.array(resp.data[0].embedding)

def cache_lookup(query, scenario_id):
    idx = [e for e in load_index() if e.get('scenario') == scenario_id]
    if not idx: return None, 0.0
    q_emb = embed(query).reshape(1, -1)
    sims  = cosine_similarity(q_emb, np.array([e['embedding'] for e in idx]))[0]
    best  = int(np.argmax(sims))
    score = float(sims[best])
    return (idx[best], score) if score >= SEM_THRESHOLD else (None, score)

def cache_save(query, q_emb, cfg, render_path, scenario_id):
    idx = load_index()
    entry_id = hashlib.md5(f'{query}::{scenario_id}::{RUN_TS}'.encode()).hexdigest()[:10]
    cfg_path = CACHE_DIR / f'{entry_id}.json'
    cfg_path.write_text(json.dumps(cfg, indent=2, ensure_ascii=False, default=str), 'utf-8')
    idx.append({
        'id':          entry_id,
        'query':       query,
        'embedding':   q_emb.tolist(),
        'config_path': str(cfg_path),
        'chart_path':  str(render_path),
        'scenario':    scenario_id,
        'timestamp':   datetime.now().isoformat(),
    })
    save_index(idx)
    print(f'  Saved to cache: {entry_id}')

existing = [e for e in load_index() if e.get('scenario') == 'SA1']
print(f'Cache ready — existing SA1 entries: {len(existing)}')


In [ ]:
# ── Cell 5: 5 trials × (Baseline → +Parallel → +Cache miss → +Cache hit) ──────
# Each trial runs all 4 strategies in sequence.
# Cache is cleared before each trial so miss is always genuine.

N_TRIALS = 5

def clear_scenario_cache(scenario_id):
    """Remove all cache entries for this scenario so next lookup is a fresh miss."""
    idx = load_index()
    kept = [e for e in idx if e.get('scenario') != scenario_id]
    removed = len(idx) - len(kept)
    save_index(kept)
    if removed:
        print(f'  [cache cleared: removed {removed} {scenario_id} entries]')

print(f'SA1 Combined Strategies — {RUN_TS}')
print(f'Query   : {QUERY}')
print(f'Trials  : {N_TRIALS}  (each trial = Baseline → +Parallel → +Cache miss → +Cache hit)\n')

all_results = []

for trial in range(1, N_TRIALS + 1):
    print('='*65)
    print(f'TRIAL {trial}/{N_TRIALS}')

    # Clear S0 cache before every trial so miss is always genuine
    clear_scenario_cache(SCENARIO_ID)

    t_emb = 0.0
    Q_EMB = None

    # ── Step 1: Baseline ──────────────────────────────────────────────────────
    cold = (trial == 1)
    t0 = time.perf_counter()
    cfg1, tm1, tok1 = run_s0_pipeline(QUERY, MD_TABLE, parallel=False)
    total1 = time.perf_counter() - t0
    q1 = validate(cfg1, render=(trial==1), tag=f't{trial}_c1_baseline', save_dir=OUT_DIR/'renders')
    tag = ' [COLD]' if cold else ''
    print(f'  1. Baseline{tag}    : {total1:.2f}s  router={tm1.get("router",0):.2f}  analyze={tm1.get("analyze",0):.2f}  extract={tm1.get("extract",0):.2f}  justify={tm1.get("justify",0):.2f}  Q={q1["quality_score"]}/8')
    all_results.append({'trial':trial,'combo':'Baseline','cold_start':cold,'cache_hit':False,
        'total_latency':total1,'router_latency':tm1.get('router',0),'analyze_latency':tm1.get('analyze',0),
        'extract_latency':tm1.get('extract',0),'justify_latency':tm1.get('justify',0),
        'parallel_latency':0,'cache_lookup_latency':0,'embed_latency':0,
        'quality_score':q1['quality_score'],'passed':q1['passed'],'charttype':q1['charttype'],
        'tokens_total':tok1.get('total_tokens',0)})

    # ── Step 2: +Parallel ─────────────────────────────────────────────────────
    t0 = time.perf_counter()
    cfg2, tm2, tok2 = run_s0_pipeline(QUERY, MD_TABLE, parallel=True)
    total2 = time.perf_counter() - t0
    q2 = validate(cfg2, render=(trial==1), tag=f't{trial}_c2_parallel', save_dir=OUT_DIR/'renders')
    par_t = tm2.get('extract∥justify', tm2.get('extract||justify', 0))
    print(f'  2. +Parallel       : {total2:.2f}s  router={tm2.get("router",0):.2f}  analyze={tm2.get("analyze",0):.2f}  parallel={par_t:.2f}  Q={q2["quality_score"]}/8')
    all_results.append({'trial':trial,'combo':'+Parallel','cold_start':False,'cache_hit':False,
        'total_latency':total2,'router_latency':tm2.get('router',0),'analyze_latency':tm2.get('analyze',0),
        'extract_latency':0,'justify_latency':0,'parallel_latency':par_t,
        'cache_lookup_latency':0,'embed_latency':0,
        'quality_score':q2['quality_score'],'passed':q2['passed'],'charttype':q2['charttype'],
        'tokens_total':tok2.get('total_tokens',0)})

    # ── Embed (after LLM combos so they aren't slowed) ────────────────────────
    t0_e = time.perf_counter()
    Q_EMB = embed(QUERY)
    t_emb = time.perf_counter() - t0_e

    # ── Step 3: +Cache miss ───────────────────────────────────────────────────
    t0 = time.perf_counter()
    tl0 = time.perf_counter()
    cached3, sim3 = cache_lookup(QUERY, SCENARIO_ID)
    tl3 = time.perf_counter() - tl0
    if cached3:
        # Shouldn't happen since we cleared cache
        cfg3, tm3, tok3 = json.loads(Path(cached3['config_path']).read_text()), {}, {}
        hit3 = True
        print(f'  3. +Cache miss     : UNEXPECTED HIT')
    else:
        cfg3, tm3, tok3 = run_s0_pipeline(QUERY, MD_TABLE, parallel=False)
        cache_save(QUERY, Q_EMB, cfg3, OUT_DIR/'renders'/f't{trial}_c3_miss.png', SCENARIO_ID)
        hit3 = False
    total3 = time.perf_counter() - t0
    q3 = validate(cfg3, render=(trial==1), tag=f't{trial}_c3_miss', save_dir=OUT_DIR/'renders')
    print(f'  3. +Cache miss     : {total3:.2f}s  lookup={tl3:.3f}s  embed={t_emb:.2f}s  router={tm3.get("router",0):.2f}  analyze={tm3.get("analyze",0):.2f}  Q={q3["quality_score"]}/8')
    all_results.append({'trial':trial,'combo':'+Cache (miss)','cold_start':False,'cache_hit':hit3,
        'total_latency':total3,'router_latency':tm3.get('router',0),'analyze_latency':tm3.get('analyze',0),
        'extract_latency':tm3.get('extract',0),'justify_latency':tm3.get('justify',0),
        'parallel_latency':0,'cache_lookup_latency':tl3,'embed_latency':t_emb,
        'quality_score':q3['quality_score'],'passed':q3['passed'],'charttype':q3['charttype'],
        'tokens_total':tok3.get('total_tokens',0)})

    # ── Step 4: +Cache hit ────────────────────────────────────────────────────
    t0 = time.perf_counter()
    tl0 = time.perf_counter()
    cached4, sim4 = cache_lookup(QUERY, SCENARIO_ID)
    tl4 = time.perf_counter() - tl0
    total4 = time.perf_counter() - t0
    if cached4:
        cfg4 = json.loads(Path(cached4['config_path']).read_text())
        hit4 = True
    else:
        print(f'  4. +Cache hit      : UNEXPECTED MISS — fallback to pipeline')
        cfg4, _, _ = run_s0_pipeline(QUERY, MD_TABLE, parallel=False)
        hit4 = False
    q4 = validate(cfg4, render=(trial==1), tag=f't{trial}_c4_hit', save_dir=OUT_DIR/'renders')
    print(f'  4. +Cache hit      : {total4:.2f}s  sim={sim4:.3f}  embed={t_emb:.2f}s  lookup={tl4:.3f}s  Q={q4["quality_score"]}/8')
    all_results.append({'trial':trial,'combo':'+Cache (hit)','cold_start':False,'cache_hit':hit4,
        'total_latency':total4,'router_latency':0,'analyze_latency':0,
        'extract_latency':0,'justify_latency':0,'parallel_latency':0,
        'cache_lookup_latency':tl4,'embed_latency':t_emb,
        'quality_score':q4['quality_score'],'passed':q4['passed'],'charttype':q4['charttype'],
        'tokens_total':0})

print('\n' + '='*65)
print(f'All {N_TRIALS} trials complete  ({N_TRIALS * 4} total runs).')


In [ ]:
# ── Cell 6: Stats summary + CSV ───────────────────────────────────────────────
df = pd.DataFrame(all_results)

print('\n' + '='*75)
print('  SA1 COMBINED STRATEGIES — RAW RUNS')
print('='*75)
for combo in df['combo'].unique():
    sub = df[df['combo']==combo]
    print(f'\n  {combo}:')
    for _, r in sub.iterrows():
        cold = ' [COLD]' if r['cold_start'] else ''
        hit  = ' [HIT]'  if r['cache_hit']  else ''
        print(f"    Run {int(r['trial'])}{cold}{hit}: {r['total_latency']:.2f}s  Q={r['quality_score']}/8")

# ── Statistics per combo ──────────────────────────────────────────────────────
print('\n' + '='*75)
print('  STATISTICS PER COMBO (total latency)')
print('='*75)
print(f'  {"Combo":<22} {"N":>3}  {"Mean":>7}  {"Median":>7}  {"SD":>6}  {"P90":>7}  {"P95":>7}  {"P99":>7}  {"Min":>7}  {"Max":>7}')
print('  ' + '-'*73)

stat_rows = []
for combo in ['Baseline','+Parallel','+Cache (miss)','+Cache (hit)']:
    vals = df[df['combo']==combo]['total_latency'].values
    if len(vals) == 0: continue
    row = {
        'combo':  combo,
        'n':      len(vals),
        'mean':   round(vals.mean(), 3),
        'median': round(float(np.median(vals)), 3),
        'std':    round(vals.std(), 3),
        'p90':    round(float(np.percentile(vals, 90)), 3),
        'p95':    round(float(np.percentile(vals, 95)), 3),
        'p99':    round(float(np.percentile(vals, 99)), 3),
        'min':    round(vals.min(), 3),
        'max':    round(vals.max(), 3),
    }
    stat_rows.append(row)
    print(f"  {combo:<22} {row['n']:>3}  {row['mean']:>6.2f}s  {row['median']:>6.2f}s  "
          f"{row['std']:>5.2f}s  {row['p90']:>6.2f}s  {row['p95']:>6.2f}s  "
          f"{row['p99']:>6.2f}s  {row['min']:>6.2f}s  {row['max']:>6.2f}s")

print('='*75)
df_stats = pd.DataFrame(stat_rows)

b_mean = df_stats[df_stats['combo']=='Baseline']['mean'].values[0]
h_mean = df_stats[df_stats['combo']=='+Cache (hit)']['mean'].values[0]
p_mean = df_stats[df_stats['combo']=='+Parallel']['mean'].values[0]
print(f'\n  Baseline mean   : {b_mean:.2f}s')
print(f'  +Parallel mean  : {p_mean:.2f}s  ({b_mean/p_mean:.2f}x)')
print(f'  Cache hit mean  : {h_mean:.2f}s  ({b_mean/h_mean:.1f}x speedup)')

csv_runs  = OUT_DIR / f'combined_SA1_runs_{RUN_TS}.csv'
csv_stats = OUT_DIR / f'combined_SA1_stats_{RUN_TS}.csv'
df.to_csv(csv_runs, index=False)
df_stats.to_csv(csv_stats, index=False)
print(f'\n  Runs CSV  : {csv_runs}')
print(f'  Stats CSV : {csv_stats}')


In [ ]:
# ── Cell 7: White-background chart + separate LaTeX table ─────────────────────
combos = ['Baseline', '+Parallel', '+Cache (miss)', '+Cache (hit)']
colors = ['#d95f02', '#7570b3', '#1b9e77', '#4c78a8']
x = np.arange(len(combos))

means = [df[df['combo'] == c]['total_latency'].mean() for c in combos]
fig, ax = plt.subplots(figsize=(9, 5.2), facecolor='white')
ax.set_facecolor('white')

bars = ax.bar(x, means, 0.55, color=colors, alpha=0.92, edgecolor='#222222', linewidth=0.6)

# Overlay individual run dots without combining a measurement table into the figure.
for i, combo in enumerate(combos):
    vals = df[df['combo'] == combo]['total_latency'].values
    jitter = np.linspace(-0.12, 0.12, len(vals)) if len(vals) > 1 else np.array([0.0])
    ax.scatter(i + jitter, vals, color='#222222', edgecolor='white', linewidth=0.7,
               s=42, zorder=5, alpha=0.9)

max_mean = max(means) if means else 0
for bar, m in zip(bars, means):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + max_mean * 0.025,
            f'{m:.2f}s', ha='center', va='bottom', fontsize=10,
            fontweight='bold', color='#222222')

ax.set_xticks(x)
ax.set_xticklabels(combos, fontsize=10, color='#222222')
ax.set_ylabel('Latency (s)', fontsize=11, color='#222222')
ax.set_title('SA1 Combined Strategies: Mean Latency', fontsize=13, fontweight='bold',
             color='#222222', pad=12)
ax.grid(axis='y', color='#dddddd', linewidth=0.8, alpha=0.9)
ax.set_axisbelow(True)
ax.tick_params(axis='y', colors='#222222')
for side in ['top', 'right']:
    ax.spines[side].set_visible(False)
for side in ['left', 'bottom']:
    ax.spines[side].set_color('#444444')

fig.tight_layout()
chart_path = OUT_DIR / f'combined_SA1_latency_white_{RUN_TS}.png'
fig.savefig(chart_path, dpi=150, bbox_inches='tight', facecolor='white')
plt.show()
plt.close(fig)

# Separate measurement table for LaTeX.
latex_df = df_stats.copy()
latex_df = latex_df[['combo', 'n', 'mean', 'median', 'std', 'p95']]
latex_df = latex_df.rename(columns={
    'combo': 'Strategy',
    'n': 'N',
    'mean': 'Mean (s)',
    'median': 'Median (s)',
    'std': 'SD (s)',
    'p95': 'P95 (s)',
})
latex_df['Speedup vs. Baseline'] = latex_df['Mean (s)'].apply(
    lambda v: f'{b_mean / v:.2f}x' if v else ''
)

latex_table = latex_df.to_latex(
    index=False,
    escape=True,
    float_format=lambda v: f'{v:.2f}',
    caption='SA1 combined strategies latency summary.',
    label='tab:sa1_combined_strategies_latency',
)
latex_path = OUT_DIR / f'combined_SA1_stats_{RUN_TS}.tex'
latex_path.write_text(latex_table, encoding='utf-8')

print(f'White-background chart: {chart_path}')
print(f'LaTeX stats table     : {latex_path}')
